In [0]:
from pyspark.sql.functions import col, explode
output_path = "/Volumes/workspace/nfl/bronze_landing/"

# Read the JSON file
df_raw = spark.read.option("multiline", "true").json(f"{output_path}*.json")

# get the teams from the json file
df_nfl_teams = (df_raw
    .select(explode("sports").alias("sport")) 
    .select(explode("sport.leagues").alias("league")) # to get to the nfl
    .select(explode("league.teams").alias("team_container")) # to get to the teams  
    .select( # convert JSON to teams table
        col("team_container.team.id").alias("id"),
        col("team_container.team.displayName").alias("displayName"),
        col("team_container.team.abbreviation").alias("abbreviation"),
        col("team_container.team.color").alias("color")
    ) # create columns 
    .dropDuplicates(["id"])
)

df_nfl_teams.display()